# Step 8: 컨텍스트 압축 + 에러 복구 — 견고한 에이전트

## 학습 목표

- **auto_compact** (LLM 요약), **micro_compact** (오래된 결과 제거), **reactive compact** (에러 후 복구) 세 가지 압축 전략을 이해합니다
- 불변 State에 **transition과 recovery 카운트**를 기록하는 패턴을 배웁니다
- **계층적 복구:** collapse → compact → error surface 전략을 구현합니다

> **왜 컨텍스트 압축이 필요한가?**
> Claude의 컨텍스트 윈도우는 유한합니다 (200K 토큰). 에이전트가 파일을 읽고, 코드를 작성하고, 테스트를 실행하다 보면
> 대화 히스토리가 급격히 커집니다. 압축 없이는 긴 작업을 완료할 수 없습니다.

---

## Claude Code 소스 분석

### 8-1. 압축 전략 개요 — query.ts의 세 가지 전략

Claude Code는 컨텍스트 관리를 위해 세 가지 압축 전략을 사용합니다:

| 전략 | 트리거 | 비용 | 효과 |
|------|--------|------|------|
| **micro_compact** | 매 턴 (오래된 결과 존재 시) | 없음 (LLM 호출 X) | 작음 |
| **auto_compact** | 토큰 수 > threshold | LLM 1회 호출 | 큼 |
| **reactive_compact** | `prompt_too_long` 에러 | LLM 1회 호출 | 큼 (긴급) |

### 8-2. Auto Compact / Micro Compact (src/query.ts:414-543)

```typescript
// src/query.ts:414-450 — shouldAutoCompact()
function shouldAutoCompact(state: State): boolean {
  // 현재 토큰 수가 threshold(기본 80,000)를 넘는지 확인
  const tokenCount = estimateTokens(state.messages)
  return tokenCount >= AUTO_COMPACT_THRESHOLD
}

// src/query.ts:480-543 — doAutoCompact()
async function doAutoCompact(state: State): Promise<State> {
  // 1. 현재 대화를 LLM에게 요약하도록 요청
  const summary = await compactConversation(state.messages, {
    prompt: BASE_COMPACT_PROMPT,    // 9개 섹션의 요약 지시 프롬프트
    customInstructions: userCustom,  // 사용자 커스텀 지시
  })

  // 2. 이전 메시지를 요약으로 대체
  const compactedMessages = [
    { role: "user", content: `[이전 대화 요약]\n${summary}` },
    lastUserMessage,  // 마지막 사용자 메시지는 유지
  ]

  // 3. 새 State 반환 (불변 전환)
  return { ...state, messages: compactedMessages, compactCount: state.compactCount + 1 }
}
```

### 8-3. BASE_COMPACT_PROMPT (src/services/compact/prompt.ts)

Claude Code의 요약 프롬프트는 9개 섹션으로 구성됩니다:
1. 사용자의 원래 요청
2. 읽은/생성한/수정한 파일 목록 (경로 포함)
3. 중요한 결정과 그 이유
4. 발생한 에러와 해결 방법
5. 남은 작업 목록
6. 기술적 세부사항 (함수명, 변수명, 에러 메시지는 정확히 유지)
7. 현재 작업 디렉토리와 환경 정보
8. 사용자의 선호사항
9. 커스텀 포커스 (사용자가 지정한 경우)

### 8-4. State 타입과 복구 필드 (src/query.ts:204-217)

```typescript
// src/query.ts:204-217 — State 타입 (복구 관련 필드 강조)

type State = {
  messages: Message[]
  turnCount: number
  transition: Continue | undefined           // ★ 왜 이 턴이 끝났는가

  // ★ 복구 카운터
  maxOutputTokensRecoveryCount: number       // max_tokens 복구 횟수
  hasAttemptedReactiveCompact: boolean       // reactive compact 시도 여부

  // ★ 컴팩션 추적
  autoCompactTracking: {
    compactCount: number
    lastCompactTurn: number
  } | undefined
}
```

### 8-5. 에러 복구 전략 (src/query.ts:1085-1252)

```typescript
// src/query.ts:1085-1252 — 에러 핸들링

// 계층적 복구 전략:
// 1. max_output_tokens → 토큰 한도 증가 → "continue" 주입
// 2. prompt_too_long → reactive compact 트리거
// 3. model_error → 다른 모델로 fallback
// 4. rate_limit → 지수 백오프로 재시도
// 5. 복구 불가 → 사용자에게 에러 표시

catch (error) {
  if (error.type === 'max_output_tokens') {
    if (state.maxOutputTokensRecoveryCount < MAX_RECOVERY) {
      // 이어서 생성하도록 "continue" 메시지 주입
      state = { ...state, maxOutputTokensRecoveryCount: count + 1 }
      continue  // 루프 재시도
    }
  }
  if (error.type === 'prompt_too_long') {
    if (!state.hasAttemptedReactiveCompact) {
      // 긴급 압축 수행
      state = await doReactiveCompact(state)
      continue  // 압축 후 재시도
    }
  }
  // 복구 불가 → 에러 표시
  throw error
}
```

### 8-6. Transition 타입 (src/query/transitions.ts)

```typescript
// src/query/transitions.ts — 상태 전이 유형

type Terminal = { reason: 'completed' | 'max_turns' | 'aborted' }
type Continue = { reason: 'tool_use' | 'max_output_recovery' | 'reactive_compact' | 'error' }
type Transition = Terminal | Continue
```

**Terminal**: 루프가 끝나는 이유 (완료, 최대 턴, 중단)
**Continue**: 루프가 계속되는 이유 (도구 사용, 복구, 압축)

In [ ]:
import sys, os
sys.path.insert(0, ".")

# mini_claude의 압축 및 상태 모듈 import
from mini_claude.compaction import (
    estimate_tokens,
    auto_compact,
    micro_compact,
    should_auto_compact,
    should_micro_compact,
    COMPACT_PROMPT,
    CLEARED_MARKER,
)
from mini_claude.state import (
    AgentState,
    Transition,
    determine_transition,
    create_initial_state,
)
from mini_claude.recovery import (
    RetryConfig,
    handle_max_output_tokens,
    handle_prompt_too_long,
)

# --- estimate_tokens 동작 확인 ---
test_messages = [
    {"role": "user", "content": "Python으로 웹 서버를 만들어주세요."},
    {"role": "assistant", "content": "네, Flask를 사용하여 간단한 웹 서버를 만들겠습니다."},
]

tokens = estimate_tokens(test_messages)
print(f"=== 토큰 추정 ===")
print(f"메시지 2개의 추정 토큰 수: {tokens}")
print(f"(단어 수 기반 추정 — tiktoken이 설치되어 있으면 더 정확합니다)")
print()

# --- AgentState 구조 확인 ---
state = create_initial_state(test_messages, max_turns=20)
print(f"=== AgentState 필드 ===")
print(f"  turn_count: {state.turn_count}")
print(f"  max_turns: {state.max_turns}")
print(f"  transition: {state.transition}")
print(f"  compact_count: {state.compact_count}")
print(f"  max_output_recovery_count: {state.max_output_recovery_count}")
print(f"  error_recovery_count: {state.error_recovery_count}")
print(f"  should_continue: {state.should_continue}")

---

## Python으로 구현하기

### 구현 1: Micro Compact — LLM 호출 없이 즉시 토큰 절약

`micro_compact`는 오래된 턴의 도구 결과를 `"[cleared]"`로 교체합니다.
파일 내용이나 명령 출력 등 큰 도구 결과를 제거하면 상당한 토큰을 절약할 수 있습니다.

In [ ]:
# 긴 대화 시뮬레이션 — 도구 호출이 많은 에이전트 세션

def generate_long_conversation(num_turns=10):
    """도구 호출이 포함된 긴 대화를 생성합니다."""
    messages = [
        {"role": "user", "content": "프로젝트의 모든 Python 파일을 분석해주세요."},
    ]
    for i in range(num_turns):
        # assistant가 도구를 호출
        messages.append({
            "role": "assistant",
            "text": f"파일 {i+1}을 읽겠습니다.",
            "tool_calls": [{"id": f"call_{i}", "name": "FileRead", "input": {"file_path": f"src/module_{i}.py"}}],
        })
        # 도구 결과 (큰 파일 내용 시뮬레이션)
        fake_content = f"# module_{i}.py\n" + "\n".join(
            [f"def function_{j}():\n    '''함수 {j} — 복잡한 비즈니스 로직'''\n    pass\n" for j in range(50)]
        )
        messages.append({
            "role": "tool",
            "tool_call_id": f"call_{i}",
            "content": fake_content,
        })
    # 마지막 assistant 응답
    messages.append({
        "role": "assistant",
        "text": "모든 파일을 분석했습니다. 총 10개의 모듈이 있습니다.",
    })
    return messages

# 긴 대화 생성
long_conversation = generate_long_conversation(num_turns=10)
original_tokens = estimate_tokens(long_conversation)

print(f"=== Micro Compact 데모 ===\n")
print(f"원본 대화: {len(long_conversation)}개 메시지, ~{original_tokens} 토큰")
print(f"micro_compact 가능? {should_micro_compact(long_conversation, max_age_turns=4)}")

# micro_compact 실행 (최근 4턴만 유지)
result = micro_compact(long_conversation, max_age_turns=4)

print(f"\n압축 후: ~{result.compacted_tokens} 토큰")
print(f"절약: ~{result.original_tokens - result.compacted_tokens} 토큰 "
      f"({(1 - result.compacted_tokens/result.original_tokens)*100:.1f}% 감소)")
print(f"압축 수행됨? {result.was_compacted}")

# 오래된 도구 결과가 [cleared]로 교체되었는지 확인
cleared_count = sum(1 for m in result.messages if m.get("content") == CLEARED_MARKER)
print(f"\n[cleared]로 교체된 도구 결과: {cleared_count}개")

# 최근 턴의 결과는 유지되었는지 확인
recent_tool_results = [m for m in result.messages[-8:] if m.get("role") == "tool" and m.get("content") != CLEARED_MARKER]
print(f"유지된 최근 도구 결과: {len(recent_tool_results)}개")

### 구현 2: AgentState와 Transition — 불변 상태 전이

`mini_claude/state.py`의 `AgentState`는 복구 카운터와 전이 이유를 포함합니다.
각 전이가 상태를 어떻게 변경하는지 시뮬레이션합니다.

In [ ]:
# AgentState 전이 시뮬레이션 — 에이전트 루프의 생명주기를 추적

print("=== AgentState 전이 시뮬레이션 ===\n")

# 초기 상태
state = create_initial_state(
    [{"role": "user", "content": "코드를 리팩토링해주세요"}],
    max_turns=10,
)
print(f"[초기] turn={state.turn_count}, transition={state.transition}")

# Turn 1: 도구 호출
state = state.next_turn(
    [{"role": "assistant", "text": "파일을 읽겠습니다."}],
    transition=Transition.TOOL_USE,
)
print(f"[Turn 1] turn={state.turn_count}, transition={state.transition.value}")

# Turn 2: 또 다른 도구 호출
state = state.next_turn(
    [{"role": "assistant", "text": "파일을 수정합니다."}],
    transition=Transition.TOOL_USE,
)
print(f"[Turn 2] turn={state.turn_count}, transition={state.transition.value}")

# Turn 3: max_output 복구 발생
state = state.with_recovery(Transition.MAX_OUTPUT_RECOVERY)
print(f"[Turn 3 - 복구] transition={state.transition.value}, "
      f"max_output_recovery={state.max_output_recovery_count}")

# Turn 4: 컴팩션 발생
state = state.with_compact(list(state.messages)[:2])
print(f"[Turn 4 - 컴팩션] transition={state.transition.value}, "
      f"compact_count={state.compact_count}, last_compact_turn={state.last_compact_turn}")

# Turn 5: 완료
state = state.next_turn(
    [{"role": "assistant", "text": "리팩토링이 완료되었습니다."}],
    transition=Transition.COMPLETED,
)
print(f"[Turn 5 - 완료] transition={state.transition.value}, "
      f"should_continue={state.should_continue}")

print()

# determine_transition 함수로 전이 결정
print("=== determine_transition 테스트 ===\n")
test_state = create_initial_state([], max_turns=5)

scenarios = [
    {"has_tool_calls": True, "desc": "도구 호출 있음"},
    {"has_tool_calls": False, "desc": "도구 호출 없음"},
    {"has_tool_calls": False, "was_truncated": True, "desc": "응답 잘림 (max_tokens)"},
    {"has_tool_calls": False, "had_error": True, "desc": "에러 발생"},
    {"has_tool_calls": True, "abort_requested": True, "desc": "사용자 중단"},
]

for scenario in scenarios:
    desc = scenario.pop("desc")
    transition = determine_transition(**scenario, state=test_state)
    print(f"  {desc:30s} → {transition.value}")
    scenario["desc"] = desc  # restore

---

## 실습: 에러 복구 전략 데모

Claude Code의 에러 복구 전략을 시뮬레이션합니다:
1. `max_output_tokens` 초과 시 토큰 한도 증가 또는 "continue" 주입
2. `prompt_too_long` 에러 시 컨텍스트 압축 트리거
3. 복구 카운터가 한도를 넘으면 종료

In [ ]:
# 에러 복구 전략 데모

print("=== 에러 복구 전략 ===\n")

# --- 1. max_output_tokens 복구 ---
print("--- max_output_tokens 복구 ---")
print()

# 첫 번째 시도: 토큰 한도 증가
action = handle_max_output_tokens(
    stop_reason="max_tokens",
    current_max_tokens=4096,
    max_limit=16384,
)
print(f"  현재 한도: 4,096 → 액션: {action['action']}")
if action["action"] == "increase_limit":
    print(f"  새 한도: {action['new_max_tokens']:,}")

# 한도를 이미 최대로 올린 경우: continue 주입
action = handle_max_output_tokens(
    stop_reason="max_tokens",
    current_max_tokens=16384,
    max_limit=16384,
)
print(f"\n  현재 한도: 16,384 (최대) → 액션: {action['action']}")
if action["action"] == "inject_continue":
    print(f"  주입 메시지: {action['continue_message']['content'][:50]}...")

# stop_reason이 max_tokens가 아닌 경우
action = handle_max_output_tokens(
    stop_reason="end_turn",
    current_max_tokens=4096,
)
print(f"\n  stop_reason=end_turn → 액션: {action['action']} (복구 불필요)")

print()
print("--- prompt_too_long 복구 ---")
print()

# 프롬프트가 너무 긴 경우
action = handle_prompt_too_long(
    messages=[{"role": "user", "content": "x" * 1000}] * 100,
    max_context_tokens=100_000,
    estimated_tokens=150_000,
)
print(f"  추정 토큰: 150,000 / 한도: 100,000 → 액션: {action['action']}")

# 한도 이내인 경우
action = handle_prompt_too_long(
    messages=[],
    max_context_tokens=100_000,
    estimated_tokens=50_000,
)
print(f"  추정 토큰: 50,000 / 한도: 100,000 → 액션: {action['action']}")

print()
print("--- 복구 카운터 한도 초과 시뮬레이션 ---")
print()

state = create_initial_state([], max_turns=20)
for i in range(4):
    can_recover = state.can_recover_max_output
    print(f"  복구 시도 #{i+1}: can_recover={can_recover}, count={state.max_output_recovery_count}")
    if can_recover:
        state = state.with_recovery(Transition.MAX_OUTPUT_RECOVERY)
    else:
        print(f"  → 복구 한도 초과! (MAX_RECOVERY_ATTEMPTS={state.MAX_RECOVERY_ATTEMPTS})")
        break

---

## 핵심 설계 원칙 정리

### 1. 계층적 압축 전략 (Tiered Compaction)
저비용(micro_compact)에서 고비용(auto_compact)으로 단계적 접근. 매 턴마다 비싼 LLM 호출을 하지 않고, 간단한 치환으로 먼저 토큰을 절약합니다. LLM 요약은 정말 필요할 때만 수행합니다.

### 2. 불변 State + 전이 기록
`AgentState`는 `frozen dataclass`로 모든 변경이 새 객체를 생성합니다. `transition` 필드로 "왜 이 턴이 끝났는가"를 기록하면, 복구 로직이 "이전에 무슨 일이 있었는지"를 알고 적절한 대응을 할 수 있습니다.

### 3. 복구 카운터로 무한 루프 방지
max_output 복구, 에러 복구 모두 최대 횟수(MAX_RECOVERY_ATTEMPTS=3)를 가집니다. 같은 에러가 반복되면 복구를 포기하고 사용자에게 알립니다. 이것이 에이전트가 무한 루프에 빠지지 않게 하는 핵심입니다.

### 4. Reactive Compact — 에러를 기회로
`prompt_too_long` 에러가 발생하면 즉시 컨텍스트를 압축하고 재시도합니다. 에러를 단순히 표시하는 게 아니라, 자동으로 복구하여 사용자 경험을 유지합니다.

---

## 다음 Step 미리보기

컨텍스트 압축으로 긴 세션을 처리할 수 있게 되었지만, 세션이 끝나면 모든 맥락이 사라집니다.

**Step 9: Memory 시스템**에서는:
- **4가지 메모리 타입** (user, feedback, project, reference)의 의미
- **MEMORY.md 인덱스 + 개별 토픽 파일** (YAML frontmatter) 구조
- 시스템 프롬프트에 **메모리를 주입**하여 세션 간 맥락을 유지하는 방법

을 배웁니다.